<a href="https://colab.research.google.com/github/yhshengjy/ClinPKPD/blob/main/Notebook4_%E4%BA%8C%E5%AE%A4%E6%A8%A1%E5%9E%8B%E4%B8%8E%E8%8D%AF%E7%89%A9%E5%88%86%E5%B8%83.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4：二室模型与药物分布

本 Notebook 是临床药学 PK/PD 交互式模拟平台中“基础药代动力学模型”部分的进阶模块。

前面的 Notebook 已经学习了：

- 一室模型中的剂量、分布容积、清除率和半衰期
- 静脉与口服给药后的浓度–时间曲线
- 多剂量给药、蓄积和稳态浓度

一室模型假设药物进入体内后迅速分布到一个均一空间中。这个假设有助于建立基础 PK 思维，但部分药物在静脉给药后会先出现较快的浓度下降，随后进入较慢的终末下降阶段。此时，仅用一个房室和一个指数过程可能无法充分描述整个浓度–时间曲线。

本节将进一步学习：

> 药物如何在中央室和外周室之间分布，以及分布过程与真正的体外清除有何区别。

本 Notebook 使用静脉推注二室模型作为教学示例，主要关注：

- 中央室和外周室
- 消除清除率 $CL$
- 房室间清除率 $Q$
- 中央室容积 $V_c$ 和外周室容积 $V_p$
- 快速分布相与较慢终末相
- 采样时间对模型识别的影响

本节也为后续“非房室分析（NCA）”做好准备，因为学生需要先理解早期分布相和终末相，才能合理解释终末半衰期和采样设计。

**重要说明：**

本 Notebook 是教学模拟，不是具体药物或患者的临床剂量计算工具。示例参数用于帮助理解模型结构，不代表任何特定药物的推荐参数。


## 1. 学习目标

完成本 Notebook 后，你应该能够：

1. 区分中央室、外周室、消除清除率和房室间清除率。
2. 解释静脉给药后快速分布相与较慢终末相的形成原因。
3. 描述 $V_c$、$V_p$、$CL$ 和 $Q$ 对浓度–时间曲线的主要影响。
4. 说明为什么 $Q$ 表示房室间交换，而不是药物从体内被清除。
5. 解释早期和晚期采样点对二室模型识别的重要性。


## 2. 为什么需要二室模型？

在一室模型中，人体被简化为一个“立即混合均匀的容器”。静脉推注后，药物浓度按一个指数过程下降：

$$
C(t)=C_0e^{-kt}
$$

但是，真实药物进入血液后，往往先分布到血流丰富、平衡较快的组织，再逐渐进入平衡较慢的组织。因此，静脉给药后的浓度下降可能包括两个不同过程：

1. **早期快速下降**：既包含药物从中央室向外周室分布，也包含消除。
2. **后期较慢下降**：中央室和外周室逐渐接近平衡，终末过程由多个参数共同决定。

二室模型并不是把人体机械地分成两个具体器官，而是用两个数学空间近似描述不同速度的分布过程。

可以这样理解：

- **中央室**：血浆和与血液快速平衡的组织。
- **外周室**：与中央室交换相对较慢的组织。
- **$Q$**：药物在两个房室之间交换的能力。
- **$CL$**：药物从中央室被真正清除出体外的能力。

二室模型的重要价值不是“参数更多”，而是帮助我们认识：

> 浓度下降不一定全部来自消除，早期浓度下降可能主要反映分布。


## 3. 二室模型的基本结构

设中央室药物量为 $A_c$，外周室药物量为 $A_p$，则：

$$
C_c=\frac{A_c}{V_c}
$$

$$
C_p=\frac{A_p}{V_p}
$$

静脉推注后，中央室和外周室药物量的变化可写为：

$$
\frac{dA_c}{dt}
=
-CL\frac{A_c}{V_c}
-Q\left(\frac{A_c}{V_c}-\frac{A_p}{V_p}\right)
$$

$$
\frac{dA_p}{dt}
=
Q\left(\frac{A_c}{V_c}-\frac{A_p}{V_p}\right)
$$

变量含义：

| 符号 | 含义 | 常用单位 |
|---|---|---|
| $Dose$ | 静脉推注剂量 | mg |
| $V_c$ | 中央室容积 | L |
| $V_p$ | 外周室容积 | L |
| $CL$ | 药物从中央室被清除的能力 | L/h |
| $Q$ | 中央室与外周室之间的房室间清除率 | L/h |
| $A_c$、$A_p$ | 两个房室中的药物量 | mg |
| $C_c$、$C_p$ | 两个房室中的药物浓度 | mg/L |

给药后即刻，药物首先位于中央室，因此：

$$
C_c(0)=\frac{Dose}{V_c}
$$

这解释了为什么静脉推注后的初始中央室浓度主要由剂量和 $V_c$ 决定。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

TRAPEZOID = getattr(np, "trapezoid", np.trapz)

from scipy.integrate import solve_ivp


def two_compartment_iv_bolus(t, dose_mg, vc_l, vp_l, cl_l_h, q_l_h):
    """Return central/peripheral amounts and concentrations after an IV bolus."""
    t = np.asarray(t, dtype=float)

    def rhs(_, amounts):
        a_c, a_p = amounts
        c_c = a_c / vc_l
        c_p = a_p / vp_l
        da_c = -cl_l_h * c_c - q_l_h * (c_c - c_p)
        da_p = q_l_h * (c_c - c_p)
        return [da_c, da_p]

    solution = solve_ivp(
        rhs,
        (float(t.min()), float(t.max())),
        [dose_mg, 0.0],
        t_eval=t,
        rtol=1e-8,
        atol=1e-10,
    )
    a_c, a_p = solution.y
    return a_c, a_p, a_c / vc_l, a_p / vp_l


def hybrid_constants(vc_l, vp_l, cl_l_h, q_l_h):
    """Calculate alpha, beta, and their half-lives for a two-compartment model."""
    k10 = cl_l_h / vc_l
    k12 = q_l_h / vc_l
    k21 = q_l_h / vp_l
    total = k10 + k12 + k21
    disc = max(total**2 - 4 * k10 * k21, 0.0)
    alpha = 0.5 * (total + np.sqrt(disc))
    beta = 0.5 * (total - np.sqrt(disc))
    return alpha, beta, np.log(2) / alpha, np.log(2) / beta


def one_compartment_iv_bolus(t, dose_mg, vd_l, cl_l_h):
    return dose_mg / vd_l * np.exp(-(cl_l_h / vd_l) * np.asarray(t))


## 4. 交互模拟 1：中央室、外周室与药物量变化

下面的模拟同时显示：

- 中央室浓度
- 外周室浓度
- 两个房室中的药物量
- 体内剩余药物量
- 快速相和终末相的半衰期

你可以调节：

- Dose：静脉推注剂量
- Vc：中央室容积
- Vp：外周室容积
- CL：消除清除率
- Q：房室间清除率
- Time：观察时间
- Log scale：是否使用对数纵坐标

建议先保持其他参数不变，每次只改变一个参数。

请重点观察：

- 给药后中央室浓度是否立即达到最高值？
- 外周室浓度为什么从 0 开始升高？
- 增大 $Q$ 后，中央室早期浓度下降是否更快？
- 改变 $CL$ 与改变 $Q$ 对“体内剩余药物量”的影响是否相同？


In [ ]:
def plot_two_compartment_profile(
    dose_mg=500,
    vc_l=15,
    vp_l=35,
    cl_l_h=5,
    q_l_h=8,
    time_h=24,
    log_scale=False,
):
    t = np.linspace(0, time_h, 700)
    a_c, a_p, c_c, c_p = two_compartment_iv_bolus(
        t, dose_mg, vc_l, vp_l, cl_l_h, q_l_h
    )
    alpha, beta, t_half_alpha, t_half_beta = hybrid_constants(
        vc_l, vp_l, cl_l_h, q_l_h
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t, c_c, linewidth=2, label="Central concentration")
    ax.plot(t, c_p, linewidth=2, linestyle="--", label="Peripheral concentration")
    ax.set_title("Two-Compartment IV Bolus Model")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    if log_scale:
        ax.set_yscale("log")
        ax.set_ylim(bottom=max(np.min(c_c[c_c > 0]) * 0.7, 1e-3))
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(t, a_c / dose_mg * 100, linewidth=2, label="Amount in central compartment")
    ax.plot(t, a_p / dose_mg * 100, linewidth=2, linestyle="--", label="Amount in peripheral compartment")
    ax.plot(t, (a_c + a_p) / dose_mg * 100, linewidth=2, linestyle=":", label="Amount remaining in body")
    ax.set_title("Drug Amount Across Compartments")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Percent of dose (%)")
    ax.set_ylim(0, 105)
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Dose", "Central volume (Vc)", "Peripheral volume (Vp)",
            "Elimination clearance (CL)", "Distribution clearance (Q)",
            "Alpha half-life", "Beta half-life", "Initial central concentration"
        ],
        "Value": [
            f"{dose_mg:.0f} mg", f"{vc_l:.1f} L", f"{vp_l:.1f} L",
            f"{cl_l_h:.1f} L/h", f"{q_l_h:.1f} L/h",
            f"{t_half_alpha:.2f} h", f"{t_half_beta:.2f} h",
            f"{dose_mg / vc_l:.2f} mg/L"
        ]
    })
    display(summary)


interact(
    plot_two_compartment_profile,
    dose_mg=FloatSlider(value=500, min=100, max=1500, step=50, description="Dose"),
    vc_l=FloatSlider(value=15, min=5, max=50, step=2.5, description="Vc"),
    vp_l=FloatSlider(value=35, min=5, max=100, step=5, description="Vp"),
    cl_l_h=FloatSlider(value=5, min=1, max=15, step=0.5, description="CL"),
    q_l_h=FloatSlider(value=8, min=0.5, max=30, step=0.5, description="Q"),
    time_h=Dropdown(options=[8, 12, 24, 48], value=24, description="Time"),
    log_scale=Checkbox(value=False, description="Log scale"),
);


interactive(children=(FloatSlider(value=500.0, description='Dose', max=1500.0, min=100.0, step=50.0), FloatSli…

## 5. 观察任务 1：理解二室分布过程

请使用上面的交互模拟完成以下任务。

### 任务 A：标准参数

设置：

- Dose = 500 mg
- Vc = 15 L
- Vp = 35 L
- CL = 5 L/h
- Q = 8 L/h
- Time = 24 h

记录：

- Initial central concentration
- Alpha half-life
- Beta half-life

观察中央室浓度和外周室浓度的变化方向。

### 任务 B：改变房室间清除率

保持其他参数不变，分别设置：

- Q = 2 L/h
- Q = 8 L/h
- Q = 24 L/h

观察：

- Q 增大时，早期中央室浓度是否下降更快？
- 外周室浓度是否更快升高？
- Q 增大是否等同于药物更快从体内消失？

### 任务 C：改变中央室容积

将 Vc 从 15 L 改为 30 L。

观察：

- 初始中央室浓度如何变化？
- 为什么 $C_c(0)=Dose/V_c$？
- Vc 改变后，早期曲线是否明显变化？

### 任务 D：比较 CL 和 Q

分别单独增大 CL 与 Q。

观察：

- 增大 CL 时，体内剩余药物量是否更快减少？
- 增大 Q 时，药物是否主要在两个房室之间重新分配？


## 6. 分布相、终末相与混合速率常数

二室模型静脉推注后的中央室浓度常写为：

$$
C_c(t)=Ae^{-\alpha t}+Be^{-\beta t}
$$

其中：

$$
\alpha>\beta
$$

通常可以把两个阶段直观理解为：

- **$\alpha$ 相**：给药后的较快下降阶段，分布过程影响明显。
- **$\beta$ 相**：较慢的终末下降阶段。

相应半衰期为：

$$
t_{1/2,\alpha}=\frac{0.693}{\alpha}
$$

$$
t_{1/2,\beta}=\frac{0.693}{\beta}
$$

需要特别注意：

- $\alpha$ 和 $\beta$ 是由 $V_c$、$V_p$、$CL$ 和 $Q$ 共同决定的混合速率常数。
- $\alpha$ 不能简单等同于 $Q$。
- $\beta$ 也不能简单等同于 $CL/V$。
- 终末半衰期变长，并不一定只代表清除率下降，也可能与分布容积和房室交换有关。

因此，在二室模型中，不能只用一个半衰期解释全部药物过程。


## 7. 交互模拟 2：房室间清除率 $Q$ 的影响

下面同时比较较低、参考和较高的 $Q$。

为了突出分布过程，请重点观察给药后的早期中央室浓度，而不要只看最后几个时间点。

请思考：

- 当 $Q$ 较低时，药物进入外周室是否较慢？
- 当 $Q$ 较高时，中央室早期浓度是否更快下降？
- 三条曲线在后期是否可能逐渐接近？
- $Q$ 改变时，$\alpha$ 半衰期和 $\beta$ 半衰期是否按相同比例变化？


In [ ]:
def plot_distribution_clearance_comparison(
    dose_mg=500,
    vc_l=15,
    vp_l=35,
    cl_l_h=5,
    q_l_h=8,
    time_h=12,
):
    t = np.linspace(0, time_h, 600)
    q_values = [max(q_l_h / 3, 0.1), q_l_h, q_l_h * 3]
    labels = ["Lower Q", "Reference Q", "Higher Q"]

    fig, ax = plt.subplots(figsize=(9, 5))
    rows = []
    for q_value, label in zip(q_values, labels):
        _, _, c_c, _ = two_compartment_iv_bolus(
            t, dose_mg, vc_l, vp_l, cl_l_h, q_value
        )
        alpha, beta, h_alpha, h_beta = hybrid_constants(vc_l, vp_l, cl_l_h, q_value)
        ax.plot(t, c_c, linewidth=2, label=f"{label}: Q={q_value:.1f} L/h")
        rows.append({
            "Scenario": label,
            "Q_L_h": q_value,
            "Alpha_half_life_h": h_alpha,
            "Beta_half_life_h": h_beta,
            "Concentration_at_1h_mg_L": np.interp(1, t, c_c),
        })

    ax.set_title("Effect of Distribution Clearance on Central Concentration")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Central concentration (mg/L)")
    ax.legend()
    plt.show()
    display(pd.DataFrame(rows).round(3))


interact(
    plot_distribution_clearance_comparison,
    dose_mg=FloatSlider(value=500, min=100, max=1500, step=50, description="Dose"),
    vc_l=FloatSlider(value=15, min=5, max=50, step=2.5, description="Vc"),
    vp_l=FloatSlider(value=35, min=5, max=100, step=5, description="Vp"),
    cl_l_h=FloatSlider(value=5, min=1, max=15, step=0.5, description="CL"),
    q_l_h=FloatSlider(value=8, min=1, max=20, step=1, description="Q"),
    time_h=Dropdown(options=[6, 12, 24], value=12, description="Time"),
);


interactive(children=(FloatSlider(value=500.0, description='Dose', max=1500.0, min=100.0, step=50.0), FloatSli…

## 8. 观察任务 2：分布交换不等于体外清除

### 任务 A：标准比较

设置：

- Dose = 500 mg
- Vc = 15 L
- Vp = 35 L
- CL = 5 L/h
- Q = 8 L/h
- Time = 12 h

记录三种情景下：

- Concentration at 1 h
- Alpha half-life
- Beta half-life

### 任务 B：增加 Q

观察 Higher Q 曲线。

思考：

- 给药后 1 小时中央室浓度为什么降低？
- 药物是被清除出体外，还是更多进入了外周室？
- 如果只测量中央室浓度，是否可能把分布误认为消除？

### 任务 C：降低 CL

保持 Q 不变，将 CL 降低。

观察：

- 终末阶段是否变慢？
- 体内药物量是否更长时间保留？
- 这与单纯改变 Q 有何不同？


## 9. 采样设计为什么会影响模型判断？

模型能否被识别，不仅取决于模型本身，也取决于采样时间。

如果缺少给药后的早期样本：

- 快速分布相可能无法被观察到。
- 二室曲线可能看起来接近一室模型。
- $V_c$ 和 $Q$ 可能难以可靠估计。

如果只有早期样本而缺少晚期样本：

- 终末相可能无法被充分描述。
- 终末半衰期可能被错误外推。
- 后期暴露预测可能不可靠。

下面先用二室模型生成“真实曲线”，再使用一室对数线性模型强行拟合不同采样设计。这个演示不是正式模型选择过程，而是帮助理解：

> 曲线通过几个观测点，并不代表模型结构一定正确。


In [ ]:
def plot_one_vs_two_compartment_sampling(
    dose_mg=500,
    vc_l=15,
    vp_l=35,
    cl_l_h=5,
    q_l_h=8,
    sampling_design="Rich sampling",
):
    schedules = {
        "Rich sampling": np.array([0.08, 0.17, 0.33, 0.5, 1, 2, 4, 8, 12, 24]),
        "No early samples": np.array([2, 4, 8, 12, 24]),
        "Early samples only": np.array([0.08, 0.17, 0.33, 0.5, 1, 2, 4]),
    }
    samples = schedules[sampling_design]
    t = np.linspace(0.02, 24, 800)
    _, _, true_curve, _ = two_compartment_iv_bolus(
        t, dose_mg, vc_l, vp_l, cl_l_h, q_l_h
    )
    _, _, observed, _ = two_compartment_iv_bolus(
        samples, dose_mg, vc_l, vp_l, cl_l_h, q_l_h
    )

    slope, intercept = np.polyfit(samples, np.log(observed), 1)
    k_fit = max(-slope, 1e-6)
    c0_fit = np.exp(intercept)
    one_comp_fit = c0_fit * np.exp(-k_fit * t)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t, true_curve, linewidth=2, label="True two-compartment curve")
    ax.plot(t, one_comp_fit, linewidth=2, linestyle="--", label="One-compartment log-linear fit")
    ax.scatter(samples, observed, s=45, label="Sampling times")
    ax.set_title(f"Sampling Design and Model Mismatch: {sampling_design}")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Central concentration (mg/L)")
    ax.set_yscale("log")
    ax.legend()
    plt.show()

    predicted_at_samples = c0_fit * np.exp(-k_fit * samples)
    rmse_log = np.sqrt(np.mean((np.log(observed) - np.log(predicted_at_samples))**2))
    display(pd.DataFrame({
        "Metric": ["Fitted one-compartment half-life", "Log-scale RMSE", "Number of samples"],
        "Value": [f"{np.log(2)/k_fit:.2f} h", f"{rmse_log:.3f}", str(len(samples))]
    }))


interact(
    plot_one_vs_two_compartment_sampling,
    dose_mg=FloatSlider(value=500, min=100, max=1500, step=50, description="Dose"),
    vc_l=FloatSlider(value=15, min=5, max=50, step=2.5, description="Vc"),
    vp_l=FloatSlider(value=35, min=5, max=100, step=5, description="Vp"),
    cl_l_h=FloatSlider(value=5, min=1, max=15, step=0.5, description="CL"),
    q_l_h=FloatSlider(value=8, min=0.5, max=30, step=0.5, description="Q"),
    sampling_design=Dropdown(
        options=["Rich sampling", "No early samples", "Early samples only"],
        value="Rich sampling",
        description="Design"
    ),
);


interactive(children=(FloatSlider(value=500.0, description='Dose', max=1500.0, min=100.0, step=50.0), FloatSli…

## 10. 观察任务 3：早期与晚期采样

分别选择以下采样设计：

- Rich sampling
- No early samples
- Early samples only

### 任务 A：丰富采样

观察一室拟合曲线能否同时描述早期和晚期数据。

### 任务 B：缺少早期样本

观察：

- 快速分布相是否不明显？
- 一室模型是否可能看起来“拟合尚可”？
- 缺少早期样本时，为什么难以识别 $Q$ 和 $V_c$？

### 任务 C：只有早期样本

观察：

- 一室拟合如何外推 8–24 h 浓度？
- 终末过程是否可能被明显误判？
- 为什么终末半衰期需要晚期样本支持？

### 任务 D：连接后续 NCA

思考：

- NCA 估计终末斜率时，为什么必须合理选择终末相点？
- 如果把分布相样本误当作终末相，半衰期可能如何偏倚？


## 11. 模型适用范围和局限性

1. 二室模型是数学近似，中央室和外周室不等同于两个固定解剖器官。
2. 本模块只模拟静脉推注和线性清除，未包括口服吸收、静脉输注、非线性消除或时间变化参数。
3. 示例参数用于教学，不代表某一药物或患者人群的真实参数。
4. 本模块未进行参数估计、模型比较、残差分析或外部验证。
5. 真实模型选择需要结合采样设计、数据质量、诊断图、先验药理知识和临床用途。


## 12. 自测题：二室模型与药物分布

请先独立作答，再查看下一单元格中的参考答案。

---

### 题目 1

静脉推注后中央室初始浓度最直接由哪两个因素决定？

A. $Dose$ 和 $V_c$  
B. $Q$ 和 $V_p$  
C. $CL$ 和 MIC  
D. $T_{max}$ 和生物利用度  

---

### 题目 2

房室间清除率 $Q$ 增大最直接表示什么？

A. 药物肾清除增强  
B. 中央室与外周室之间交换加快  
C. 口服吸收加快  
D. MIC 降低  

---

### 题目 3

缺少给药后的早期采样最可能导致什么问题？

A. 难以识别快速分布相  
B. AUC 必然为 0  
C. 剂量无法记录  
D. 清除率必然升高  

---

### 题目 4

关于终末半衰期，哪项说法最合理？

A. 只由 CL 决定  
B. 只由 Q 决定  
C. 由多个模型参数共同决定  
D. 与采样时间无关


## 13. 自测题参考答案

### 题目 1

**参考答案：A**

**解析：**  
静脉推注后药物首先进入中央室，因此：

$$
C_c(0)=\frac{Dose}{V_c}
$$

---

### 题目 2

**参考答案：B**

**解析：**  
$Q$ 描述两个房室之间的双向交换，不代表药物已经被清除出体外。

---

### 题目 3

**参考答案：A**

**解析：**  
快速分布相主要出现在给药后的早期阶段。缺少早期样本时，二室曲线可能看起来接近单指数下降。

---

### 题目 4

**参考答案：C**

**解析：**  
二室模型的终末过程由 $V_c$、$V_p$、$CL$ 和 $Q$ 共同影响，不能只根据一个参数解释。


## 14. 本 Notebook 小结

本 Notebook 使用静脉推注二室模型介绍了药物分布与消除的区别。

你应该掌握以下核心结论：

1. 二室模型用中央室、外周室、$CL$ 和 $Q$ 描述药物分布与消除。
2. 静脉给药后的早期浓度下降可能主要反映分布，而不只是体外清除。
3. $Q$ 决定房室间交换速度，$CL$ 决定药物从体内被清除的能力。
4. 快速相和终末相由多个微观参数共同决定，不能机械地对应单一生理过程。
5. 早期和晚期采样共同决定能否识别分布相、终末相和合理的模型结构。

本节与后续 NCA 的联系可以概括为：

$$
模型结构理解
\rightarrow
识别分布相与终末相
\rightarrow
合理选择采样时间
\rightarrow
解释终末半衰期
$$


## 15. 参考资料

1. Mould DR, Upton RN. Basic concepts in population modeling, simulation, and model-based drug development. *CPT Pharmacometrics Syst Pharmacol.* 2012;1:e6.  
2. Toutain PL, Bousquet-Mélou A. Volumes of distribution. *J Vet Pharmacol Ther.* 2004;27:441–453.  
3. Rowland M, Tozer TN. *Clinical Pharmacokinetics and Pharmacodynamics: Concepts and Applications.*  
4. Gabrielsson J, Weiner D. *Pharmacokinetic and Pharmacodynamic Data Analysis: Concepts and Applications.*
